# Analyse des tailles du dataset — Pipeline ESS Numérique\n\nObjectif : mesurer la volumétrie à chaque étape du pipeline `src/` et produire des tableaux croisés par source et traitement appliqué.\n\n**Étapes du pipeline :**\n1. **Candidats** — 4 scripts de filtrage indépendants\n2. **Consolidation** — Fusion par SIREN unique\n3. **Enrichissement** — Complétion depuis SIRENE\n4. **Export** — Nettoyage pour Label Studio

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().resolve()
# Remonter jusqu'à la racine du projet si on est dans notebooks/
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

CAND = ROOT / "data" / "processed" / "candidates"
EXPORT = ROOT / "data" / "processed" / "export"

# Fichiers du pipeline
FILES = {
    # Étape 1 : Candidats
    "ess_x_naf":        CAND / "01_ess_x_naf.csv",
    "cate_x_naf":       CAND / "02_cate_x_naf.csv",
    "tiers_lieux":      CAND / "03_tiers_lieux.csv",
    "ess_flag_insee":   CAND / "04_ess_flag_insee.csv",
    # Étape 2 : Consolidation
    "x_consolidated":   CAND / "x_consolidated.csv",
    # Étape 3 : Enrichissement
    "x_enriched":       CAND / "x_enriched.csv",
    # Étape 4 : Export
    "x_label_studio":   EXPORT / "x_label_studio.csv",
}

# Chargement
datasets = {}
for name, path in FILES.items():
    if path.exists():
        datasets[name] = pd.read_csv(path, dtype={"siren": str})
    else:
        print(f"⚠ Fichier absent : {name} ({path.name})")

print(f"Racine projet : {ROOT}")
print(f"Fichiers chargés : {len(datasets)}/{len(FILES)}")

## 1. Volumétrie par étape du pipeline\n\nNombre de lignes, SIREN uniques et colonnes à chaque fichier de sortie.

In [ ]:
ETAPE_MAP = {
    "ess_x_naf":       "1. Candidats",
    "cate_x_naf":      "1. Candidats",
    "tiers_lieux":     "1. Candidats",
    "ess_flag_insee":  "1. Candidats",
    "x_consolidated":  "2. Consolidation",
    "x_enriched":      "3. Enrichissement",
    "x_label_studio":  "4. Export",
}

rows = []
for name, df in datasets.items():
    n_siren = df["siren"].nunique() if "siren" in df.columns else None
    rows.append({
        "Étape": ETAPE_MAP.get(name, "?"),
        "Fichier": name,
        "Lignes": len(df),
        "SIREN uniques": n_siren,
        "Colonnes": len(df.columns),
    })

df_vol = pd.DataFrame(rows)
df_vol.style.format({"Lignes": "{:,}", "SIREN uniques": "{:,}"}).hide(axis="index")

### Résumé visuel : entonnoir du pipeline

In [ ]:
import matplotlib.pyplot as plt

# Somme des lignes candidats vs consolidé vs enrichi vs export
candidats_names = ["ess_x_naf", "cate_x_naf", "tiers_lieux", "ess_flag_insee"]
n_candidats_total = sum(len(datasets[n]) for n in candidats_names if n in datasets)
n_candidats_siren = pd.concat(
    [datasets[n]["siren"] for n in candidats_names if n in datasets]
).nunique()

etapes = ["Candidats\n(somme brute)", "Candidats\n(SIREN uniques)", "Consolidé", "Enrichi", "Export"]
valeurs = [
    n_candidats_total,
    n_candidats_siren,
    len(datasets["x_consolidated"]) if "x_consolidated" in datasets else 0,
    len(datasets["x_enriched"]) if "x_enriched" in datasets else 0,
    len(datasets["x_label_studio"]) if "x_label_studio" in datasets else 0,
]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(etapes[::-1], valeurs[::-1], color=["#2ecc71", "#3498db", "#3498db", "#e67e22", "#e74c3c"])
ax.set_xlabel("Nombre de lignes / SIREN")
ax.set_title("Entonnoir du pipeline ESS Numérique")
for bar, val in zip(bars, valeurs[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2, f"{val:,}", va="center", fontsize=10)
plt.tight_layout()
plt.show()

## 2. Tableau croisé : Sources × Fichier candidat\n\nDistribution de la colonne `sources` dans chaque fichier candidat.

In [ ]:
# Tableau croisé : sources × fichier candidat
rows_src = []
for name in candidats_names:
    if name not in datasets:
        continue
    df = datasets[name]
    if "sources" not in df.columns:
        continue
    for src_val, count in df["sources"].value_counts().items():
        rows_src.append({"Fichier": name, "Sources": src_val, "Lignes": count})

df_src = pd.DataFrame(rows_src)
if not df_src.empty:
    cross_src = df_src.pivot_table(index="Sources", columns="Fichier", values="Lignes", aggfunc="sum", fill_value=0)
    cross_src["TOTAL"] = cross_src.sum(axis=1)
    cross_src.loc["TOTAL"] = cross_src.sum()
    cross_src = cross_src.astype(int)
    display(cross_src.style.format("{:,}").set_caption("Nombre de lignes par source × fichier candidat"))

## 3. Tableau croisé : Méthode de jointure × Fichier candidat

In [ ]:
# Tableau croisé : méthode de jointure × fichier candidat
rows_mj = []
for name in candidats_names:
    if name not in datasets:
        continue
    df = datasets[name]
    if "methode_jointure" not in df.columns:
        continue
    for mj, count in df["methode_jointure"].value_counts().items():
        rows_mj.append({"Fichier": name, "Méthode jointure": mj, "Lignes": count})

df_mj = pd.DataFrame(rows_mj)
if not df_mj.empty:
    cross_mj = df_mj.pivot_table(index="Méthode jointure", columns="Fichier", values="Lignes", aggfunc="sum", fill_value=0)
    cross_mj["TOTAL"] = cross_mj.sum(axis=1)
    cross_mj.loc["TOTAL"] = cross_mj.sum()
    cross_mj = cross_mj.astype(int)
    display(cross_mj.style.format("{:,}").set_caption("Nombre de lignes par méthode de jointure × fichier candidat"))

## 4. Fichier consolidé : Origines × Sources\n\nTableau croisé sur le fichier `x_consolidated.csv` montrant les combinaisons origines (fichier candidat d'où vient le SIREN) et sources (provenance des données).

In [ ]:
if "x_consolidated" in datasets:
    df_cons = datasets["x_consolidated"]
    
    # Origines × Sources
    cross_os = pd.crosstab(df_cons["origines"], df_cons["sources"], margins=True, margins_name="TOTAL")
    display(cross_os.style.format("{:,}").set_caption("x_consolidated : Origines × Sources"))
    
    print(f"\nSIREN multi-origines : {df_cons['origines'].str.contains('|', regex=False).sum():,} / {len(df_cons):,}")

## 5. Consolidé : Répartition par nombre d'origines\n\nCombien de SIREN proviennent d'1, 2, 3 ou 4 fichiers candidats ?

In [ ]:
if "x_consolidated" in datasets:
    df_cons = datasets["x_consolidated"]
    n_origines = df_cons["origines"].str.count(r"\|") + 1
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Bar chart
    counts = n_origines.value_counts().sort_index()
    counts.plot.bar(ax=axes[0], color="#3498db", edgecolor="white")
    axes[0].set_xlabel("Nombre d'origines")
    axes[0].set_ylabel("SIREN")
    axes[0].set_title("SIREN par nombre d'origines")
    for i, (x, v) in enumerate(counts.items()):
        axes[0].text(i, v + 10, f"{v:,}", ha="center", fontsize=10)
    
    # Détail des origines multi-sources
    multi = df_cons[n_origines > 1]["origines"].value_counts()
    if not multi.empty:
        multi.plot.barh(ax=axes[1], color="#e67e22", edgecolor="white")
        axes[1].set_xlabel("SIREN")
        axes[1].set_title("Combinaisons multi-origines")
    
    plt.tight_layout()
    plt.show()

## 6. Enrichissement : Taux de remplissage avant/après\n\nComparaison du pourcentage de champs non-vides entre `x_consolidated` et `x_enriched` pour les colonnes clés.

In [ ]:
KEY_COLS = [
    "denomination", "naf", "categorie_juridique", "famille_ess",
    "tranche_effectifs", "code_postal", "commune", "site_web",
    "date_creation", "est_active", "siret_siege", "rna",
]

if "x_consolidated" in datasets and "x_enriched" in datasets:
    df_before = datasets["x_consolidated"]
    df_after = datasets["x_enriched"]
    
    rows_fill = []
    for col in KEY_COLS:
        pct_before = (df_before[col].notna() & (df_before[col].astype(str).str.strip() != "")).mean() * 100 if col in df_before.columns else 0
        pct_after = (df_after[col].notna() & (df_after[col].astype(str).str.strip() != "")).mean() * 100 if col in df_after.columns else 0
        rows_fill.append({
            "Champ": col,
            "Avant (consolidé)": f"{pct_before:.1f}%",
            "Après (enrichi)": f"{pct_after:.1f}%",
            "Gain": f"+{pct_after - pct_before:.1f}pp",
            "_gain": pct_after - pct_before,
        })
    
    df_fill = pd.DataFrame(rows_fill)
    display(df_fill.drop(columns="_gain").style.hide(axis="index").set_caption("Taux de remplissage avant/après enrichissement"))
    
    # Graphique
    fig, ax = plt.subplots(figsize=(10, 5))
    y = range(len(KEY_COLS))
    before_vals = [float(r["Avant (consolidé)"].rstrip("%")) for r in rows_fill]
    after_vals = [float(r["Après (enrichi)"].rstrip("%")) for r in rows_fill]
    
    ax.barh(y, after_vals, color="#2ecc71", label="Après enrichissement", alpha=0.8)
    ax.barh(y, before_vals, color="#e74c3c", label="Avant enrichissement", alpha=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(KEY_COLS)
    ax.set_xlabel("% rempli")
    ax.set_title("Taux de remplissage : consolidé vs enrichi")
    ax.legend()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 7. Répartition par code NAF et famille ESS (consolidé)

In [ ]:
if "x_consolidated" in datasets:
    df_cons = datasets["x_consolidated"]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # NAF
    if "naf" in df_cons.columns:
        naf_counts = df_cons["naf"].value_counts().head(15)
        naf_counts.plot.barh(ax=axes[0], color="#3498db", edgecolor="white")
        axes[0].set_title("Top 15 codes NAF (consolidé)")
        axes[0].set_xlabel("SIREN")
        axes[0].invert_yaxis()
    
    # Famille ESS
    if "famille_ess" in df_cons.columns:
        fam_counts = df_cons["famille_ess"].value_counts()
        fam_counts.plot.pie(ax=axes[1], autopct="%1.1f%%", startangle=90)
        axes[1].set_title("Famille ESS (consolidé)")
        axes[1].set_ylabel("")
    
    plt.tight_layout()
    plt.show()

## 8. Tableau croisé : NAF × Famille ESS (enrichi)

In [ ]:
if "x_enriched" in datasets:
    df_enr = datasets["x_enriched"]
    if "naf" in df_enr.columns and "famille_ess" in df_enr.columns:
        cross_naf_fam = pd.crosstab(
            df_enr["naf"], 
            df_enr["famille_ess"].fillna("(non classé)"),
            margins=True, margins_name="TOTAL"
        )
        display(cross_naf_fam.style.format("{:,}").set_caption("NAF × Famille ESS (enrichi)"))

## 9. Conflits détectés lors de la consolidation\n\nColonnes suffixées `__origin` indiquant des valeurs divergentes entre sources pour un même SIREN.

In [ ]:
if "x_consolidated" in datasets:
    df_cons = datasets["x_consolidated"]
    conflict_cols = [c for c in df_cons.columns if "__" in c]
    
    if conflict_cols:
        # Extraire l'attribut de base depuis le nom de colonne (ex: denomination__ess_numerique → denomination)
        attrs_with_conflicts = sorted(set(c.split("__")[0] for c in conflict_cols))
        
        rows_conf = []
        for attr in attrs_with_conflicts:
            cols_attr = [c for c in conflict_cols if c.startswith(f"{attr}__")]
            n_conflicts = df_cons[cols_attr[0]].notna().sum() if cols_attr else 0
            rows_conf.append({
                "Attribut": attr,
                "SIREN en conflit": n_conflicts,
                "% du total": f"{n_conflicts / len(df_cons) * 100:.1f}%",
                "Colonnes": ", ".join(cols_attr),
            })
        
        df_conf = pd.DataFrame(rows_conf)
        display(df_conf.style.hide(axis="index").set_caption("Conflits par attribut"))
    else:
        print("Aucun conflit détecté (pas de colonnes __origin).")

## 10. Score d'enrichissement (export Label Studio)

In [ ]:
if "x_label_studio" in datasets:
    df_ls = datasets["x_label_studio"]
    if "score_enrichissement" in df_ls.columns:
        fig, ax = plt.subplots(figsize=(8, 4))
        df_ls["score_enrichissement"].hist(bins=20, ax=ax, color="#9b59b6", edgecolor="white")
        ax.set_xlabel("Score d'enrichissement")
        ax.set_ylabel("SIREN")
        ax.set_title("Distribution du score d'enrichissement (Label Studio)")
        ax.axvline(df_ls["score_enrichissement"].median(), color="red", linestyle="--", label=f"Médiane : {df_ls['score_enrichissement'].median():.2f}")
        ax.legend()
        plt.tight_layout()
        plt.show()
        
        print(df_ls["score_enrichissement"].describe().to_string())
    else:
        print("Colonne score_enrichissement absente du fichier export.")